**Voice-to-Grasp - Experiment 4, cross-participant validation**: each participant's personalised model
is evaluated on the other participants' prompts (paper Fig. 6c). Outputs were cleared for the release.

# Cross-User Adaptive Model Evaluation

Run one **trained adaptive model** (User A) against **multiple users' data** (first 140 rows each) — to measure how well the model generalises across users without any further training.

| Setting | Value |
|---|---|
| **Source model** | Adaptive model trained on User A |
| **Evaluation data** | Each target user's XLSX, rows 1–140 only |
| **Model frozen?** | ✅ Yes — no updates, pure inference |


## 0  ·  Configuration — set your paths here

In [ ]:
import os, json
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from statsmodels.stats.proportion import proportion_confint
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datetime import datetime

sns.set_theme(style="whitegrid", palette="tab10")
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 13, "axes.labelsize": 11})

# ── EDIT THESE ────────────────────────────────────────────────────────
EXPERIMENTS_ROOT = r"D:\PythonEnvs\nlpglove\Continual_Learning_Experiments"

# User whose adaptive model you want to test with
MODEL_USER = "PilotA2"

# All users you want to evaluate on — add as many as you like
# Format: { "user_id": "path/to/their/xlsx" }
DATA_USERS = {
    "P2": r"D:\PythonEnvs\nlpglove\Continual_Learning_Experiments/UserStudy_code_analysis/VoiceToGrasp_StudySheet_P2.xlsx",
    "P3": r"D:\PythonEnvs\nlpglove\Continual_Learning_Experiments/UserStudy_code_analysis/VoiceToGrasp_StudySheet_P3.xlsx",
    "P4": r"D:\PythonEnvs\nlpglove\Continual_Learning_Experiments/UserStudy_code_analysis/VoiceToGrasp_StudySheet_P4.xlsx",
    "P1": r"D:\PythonEnvs\nlpglove\Continual_Learning_Experiments/UserStudy_code_analysis/VoiceToGrasp_StudySheet_P1.xlsx",
   
    # "PilotA": r"D:\PythonEnvs\nlpglove\Continual_Learning_Experiments/UserStudy_code_analysis/VoiceToGrasp_StudySheet_PilotA.xlsx"
}

EVAL_ROWS  = 210
OUTPUT_DIR = "cross_user_results"
# ─────────────────────────────────────────────────────────────────────

ID2LABEL = {
    0: "grasp",      1: "open_hand",   2: "gentle_grasp",
    3: "tight_grasp",4: "pinch_grasp", 5: "tighten",      6: "loosen"
}
LABEL_ORDER = list(ID2LABEL.values())

MODEL_PATH = os.path.join(EXPERIMENTS_ROOT, MODEL_USER, "adaptive_model")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Model  : {MODEL_PATH}")
print(f"Output : {OUTPUT_DIR}/")
print(f"Evaluating on {len(DATA_USERS)} user(s): {list(DATA_USERS.keys())}")

## 1  ·  Load Adaptive Model (once)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(device)
model.eval()
for p in model.parameters():
    p.requires_grad = False

print(f"Model loaded : {MODEL_PATH}")
print(f"Device       : {device}")
print(f"Parameters   : {sum(p.numel() for p in model.parameters()):,}")

## 2  ·  Helper Functions

In [ ]:
def load_data(path, n_rows):
    df = pd.read_excel(path) if path.endswith(".xlsx") else pd.read_csv(path)
    print(df.head())
    col_map = {}
    for col in df.columns:
        lc = col.strip().lower()
        if lc == "prompt":
            col_map[col] = "prompt"
        elif lc in ("grasp_type", "true_grasp"):
            col_map[col] = "true_grasp"
    df = df.rename(columns=col_map)
    if "prompt" not in df.columns or "true_grasp" not in df.columns:
        raise ValueError(f"Could not find prompt/grasp columns. Found: {df.columns.tolist()}")
    df["prompt"]     = df["prompt"].astype(str).str.strip()
    df["true_grasp"] = df["true_grasp"].astype(str).str.strip()
    return df.iloc[:n_rows].reset_index(drop=True)


def predict_all(df):
    preds, confs = [], []
    for text in df["prompt"]:
        inputs = tokenizer(text, return_tensors="pt",
                           truncation=True, padding=True).to(device)
        with torch.no_grad():
            logits = model(**inputs).logits
        probs   = torch.softmax(logits, dim=1)
        pred_id = torch.argmax(probs, dim=1).item()
        preds.append(ID2LABEL[pred_id])
        confs.append(probs[0][pred_id].item())
    df["predicted_grasp"] = preds
    df["confidence"]      = confs
    df["correct"]         = df["predicted_grasp"] == df["true_grasp"]
    return df


def ece_score(confidences, correct, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece  = 0.0
    for i in range(n_bins):
        mask = (confidences >= bins[i]) & (confidences < bins[i+1])
        if mask.sum() > 0:
            ece += abs(correct[mask].mean() - confidences[mask].mean()) * mask.sum() / len(confidences)
    return ece

## 3  ·  Run Inference Across All Target Users

In [ ]:
all_results = {}   # user_id → DataFrame

for user_id, file_path in DATA_USERS.items():
    print(f"\n{'─'*50}")
    print(f"  Evaluating: {user_id}  ({file_path})")
    try:
        df = load_data(file_path, EVAL_ROWS)
        df = predict_all(df)
        acc = df["correct"].mean()
        print(f"  Accuracy   : {acc:.4f}  ({df['correct'].sum()}/{len(df)})")
        all_results[user_id] = df
    except Exception as e:
        print(f"  ⚠ Skipped — {e}")

print(f"\n✓ Done. {len(all_results)} user(s) evaluated.")

## 4  ·  Summary Table

In [ ]:
rows = []
for user_id, df in all_results.items():
    n_correct = df["correct"].sum()
    n_total   = len(df)
    acc = n_correct / n_total
    lo, hi = proportion_confint(n_correct, n_total, method="wilson")
    ece = ece_score(df["confidence"].values, df["correct"].astype(float).values)
    rows.append({
        "Data User":       user_id,
        "Accuracy":        round(acc, 4),
        "95% CI lower":    round(lo, 4),
        "95% CI upper":    round(hi, 4),
        "Mean Confidence": round(df["confidence"].mean(), 4),
        "ECE":             round(ece, 4),
        "Correct":         int(n_correct),
        "Total":           int(n_total),
    })

summary_df = pd.DataFrame(rows).set_index("Data User")
display(summary_df.style.background_gradient(subset=["Accuracy"], cmap="RdYlGn")
                        .format("{:.4f}", subset=["Accuracy","95% CI lower","95% CI upper",
                                                   "Mean Confidence","ECE"]))

## 5  ·  Accuracy Comparison Across Users

In [ ]:
users = list(all_results.keys())
accs  = [all_results[u]["correct"].mean() for u in users]
lows  = [proportion_confint(all_results[u]["correct"].sum(), len(all_results[u]), method="wilson")[0] for u in users]
highs = [proportion_confint(all_results[u]["correct"].sum(), len(all_results[u]), method="wilson")[1] for u in users]
errs  = [[a - l for a, l in zip(accs, lows)],
         [h - a for a, h in zip(accs, highs)]]

fig, ax = plt.subplots(figsize=(max(7, len(users)*2), 4))
colors = sns.color_palette("tab10", len(users))
bars = ax.bar(users, accs, color=colors, edgecolor="white", width=0.5)
ax.errorbar(users, accs, yerr=errs, fmt="none", color="black", capsize=5, linewidth=1.2)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
            f"{acc:.3f}", ha="center", fontweight="bold", fontsize=10)
ax.set_ylim(0, 1.15)
ax.set_ylabel("Accuracy")
ax.set_title(f"Cross-User Accuracy — Model trained on: {MODEL_USER}\n(error bars = 95% Wilson CI)")
ax.axhline(np.mean(accs), color="tomato", linestyle="--",
           label=f"Mean across users = {np.mean(accs):.3f}")
ax.legend()
plt.tight_layout()
plt.show()

## 6  ·  Per-Class F1 Heatmap (all users)

In [ ]:
f1_rows = {}
for user_id, df in all_results.items():
    present = [l for l in LABEL_ORDER if l in df["true_grasp"].values]
    rpt = classification_report(df["true_grasp"], df["predicted_grasp"],
                                labels=present, zero_division=0, output_dict=True)
    f1_rows[user_id] = {l: rpt.get(l, {}).get("f1-score", 0.0) for l in LABEL_ORDER}

f1_df = pd.DataFrame(f1_rows, index=LABEL_ORDER).T   # rows=users, cols=classes

fig, ax = plt.subplots(figsize=(11, max(3, len(users)*0.8 + 1)))
sns.heatmap(f1_df, annot=True, fmt=".2f", cmap="RdYlGn",
            vmin=0, vmax=1, linewidths=0.5, linecolor="white", ax=ax)
ax.set_title(f"Per-Class F1-Score — Model: {MODEL_USER}")
ax.set_xlabel("Grasp Type"); ax.set_ylabel("Data User")
plt.tight_layout()
plt.show()

## 7  ·  Confusion Matrices (one per user)

In [ ]:
n = len(all_results)
fig, axes = plt.subplots(1, n, figsize=(9*n, 7))
if n == 1:
    axes = [axes]

for ax, (user_id, df) in zip(axes, all_results.items()):
    present = [l for l in LABEL_ORDER if l in set(df["true_grasp"]) | set(df["predicted_grasp"])]
    cm = confusion_matrix(df["true_grasp"], df["predicted_grasp"], labels=present)
    disp = ConfusionMatrixDisplay(cm, display_labels=present)
    disp.plot(ax=ax, colorbar=False, cmap="Blues", xticks_rotation=40)
    acc = df["correct"].mean()
    ax.set_title(f"Data: {user_id}\nAcc={acc:.3f}")

fig.suptitle(f"Confusion Matrices — Model: {MODEL_USER}", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 8  ·  Confidence Distribution (all users)

In [ ]:
fig, axes = plt.subplots(1, n, figsize=(7*n, 4), sharey=True)
if n == 1:
    axes = [axes]

colors = sns.color_palette("tab10", n)
for ax, (user_id, df), color in zip(axes, all_results.items(), colors):
    ax.hist(df[df["correct"]]["confidence"],  bins=12, range=(0,1),
            alpha=0.75, color="green",  label=f"Correct (n={df['correct'].sum()})")
    ax.hist(df[~df["correct"]]["confidence"], bins=12, range=(0,1),
            alpha=0.75, color="tomato", label=f"Incorrect (n={(~df['correct']).sum()})")
    ax.axvline(df["confidence"].mean(), color="black", linestyle="--",
               label=f"Mean={df['confidence'].mean():.3f}")
    ax.set_xlabel("Confidence"); ax.set_ylabel("Count")
    ax.set_title(user_id)
    ax.legend(fontsize=8)

fig.suptitle(f"Confidence Distribution — Model: {MODEL_USER}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 9  ·  Misclassification Patterns (one per user)

In [ ]:
for user_id, df in all_results.items():
    errors = df[~df["correct"]]
    print(f"{user_id}: {len(errors)} errors / {len(df)} total")
    if len(errors) == 0:
        print("  No errors — skipping heatmap.")
        continue
    pivot = (errors.groupby(["true_grasp", "predicted_grasp"])
                   .size().unstack(fill_value=0)
                   .reindex(index=LABEL_ORDER, columns=LABEL_ORDER, fill_value=0))
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(pivot, annot=True, fmt="d", cmap="Oranges",
                linewidths=0.5, linecolor="white", ax=ax)
    ax.set_title(f"Misclassification — Model: {MODEL_USER}  →  Data: {user_id}")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    plt.tight_layout()
    plt.show()

## 10  ·  Save All Results

In [ ]:
saved = []
for user_id, df in all_results.items():
    # CSV
    out_csv = os.path.join(OUTPUT_DIR, f"{MODEL_USER}_on_{user_id}.csv")
    df.to_csv(out_csv, index=False)

    # JSON
    n_correct = df["correct"].sum()
    n_total   = len(df)
    acc = n_correct / n_total
    lo, hi = proportion_confint(n_correct, n_total, method="wilson")
    ece = ece_score(df["confidence"].values, df["correct"].astype(float).values)
    present = [l for l in LABEL_ORDER if l in df["true_grasp"].values]

    meta = {
        "timestamp":            str(datetime.now()),
        "model_user":           MODEL_USER,
        "data_user":            user_id,
        "eval_rows":            EVAL_ROWS,
        "accuracy":             round(acc, 4),
        "wilson_ci_95":         [round(lo, 4), round(hi, 4)],
        "mean_confidence":      round(df["confidence"].mean(), 4),
        "ece":                  round(ece, 4),
        "correct":              int(n_correct),
        "total":                int(n_total),
        "classification_report": classification_report(
            df["true_grasp"], df["predicted_grasp"],
            labels=present, zero_division=0, output_dict=True
        ),
    }
    out_json = os.path.join(OUTPUT_DIR, f"{MODEL_USER}_on_{user_id}.json")
    with open(out_json, "w") as f:
        json.dump(meta, f, indent=4)

    saved.append(out_csv)

# Combined summary CSV
summary_df.to_csv(os.path.join(OUTPUT_DIR, f"{MODEL_USER}_summary.csv"))
saved.append(os.path.join(OUTPUT_DIR, f"{MODEL_USER}_summary.csv"))

print("Saved files:")
for s in saved:
    print(f"  {s}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  STANDALONE Cross-User Accuracy Matrix
#  No dependency on any other cell — runs everything from scratch
# ══════════════════════════════════════════════════════════════════════════════
import os, json, torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report
from statsmodels.stats.proportion import proportion_confint

# ── CONFIGURE THESE ───────────────────────────────────────────────────────────
EXPERIMENTS_ROOT = r"D:\PythonEnvs\nlpglove\Continual_Learning_Experiments"
OUTPUT_DIR       = "cross_user_results"

ID2LABEL = {
    0: "grasp",       1: "open_hand",    2: "gentle_grasp",
    3: "tight_grasp", 4: "pinch_grasp",  5: "tighten",      6: "loosen"
}
LABEL_ORDER = list(ID2LABEL.values())
EVAL_ROWS   = 210

ALL_USERS = {
    "P2": r"D:\PythonEnvs\nlpglove\Continual_Learning_Experiments/UserStudy_code_analysis/VoiceToGrasp_StudySheet_P2.xlsx",
    "P3":  r"D:\PythonEnvs\nlpglove\Continual_Learning_Experiments/UserStudy_code_analysis/VoiceToGrasp_StudySheet_P3.xlsx",
    "P4": r"D:\PythonEnvs\nlpglove\Continual_Learning_Experiments/UserStudy_code_analysis/VoiceToGrasp_StudySheet_P4.xlsx",
    "P1": r"D:\PythonEnvs\nlpglove\Continual_Learning_Experiments/UserStudy_code_analysis/VoiceToGrasp_StudySheet_P1.xlsx",
    # "PilotA":    r"D:\PythonEnvs\nlpglove\Continual_Learning_Experiments/UserStudy_code_analysis/VoiceToGrasp_StudySheet_PilotA.xlsx",
}
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
user_list = list(ALL_USERS.keys())
OUTPUT_DIR = "./paper_figures"
def _save(fig, name):
    if True:
        path = os.path.join(OUTPUT_DIR, name)
        fig.savefig(path, dpi=300, format="svg", bbox_inches="tight")
        print(f"  saved -> {path}")
    plt.show()

def load_data(path, n_rows):
    df = pd.read_excel(path) if path.endswith(".xlsx") else pd.read_csv(path)
    col_map = {}
    for col in df.columns:
        lc = col.strip().lower()
        if lc == "prompt":
            col_map[col] = "prompt"
        elif lc in ("grasp_type", "true_grasp"):
            col_map[col] = "true_grasp"
    df = df.rename(columns=col_map)
    df["prompt"]     = df["prompt"].astype(str).str.strip()
    df["true_grasp"] = df["true_grasp"].astype(str).str.strip()
    return df.iloc[:n_rows].reset_index(drop=True)

def run_inference(df, tokenizer, model):
    preds, confs = [], []
    for text in df["prompt"]:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
        with torch.no_grad():
            logits = model(**inputs).logits
        probs   = torch.softmax(logits, dim=1)
        pred_id = torch.argmax(probs, dim=1).item()
        preds.append(ID2LABEL[pred_id])
        confs.append(probs[0][pred_id].item())
    df["predicted_grasp"] = preds
    df["confidence"]      = confs
    df["correct"]         = df["predicted_grasp"] == df["true_grasp"]
    return df

# ── Main loop: for each model user, evaluate on every data user ───────────────
matrix_data = {m: {d: np.nan for d in user_list} for m in user_list}

for model_u in user_list:
    model_path = os.path.join(EXPERIMENTS_ROOT, model_u, "adaptive_model")
    if not os.path.exists(model_path):
        print(f"  ⚠ Model not found for {model_u}, skipping row.")
        continue

    print(f"\nLoading model: {model_u}")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model     = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
    model.eval()
    for p in model.parameters():
        p.requires_grad = False

    for data_u in user_list:
        json_path = os.path.join(OUTPUT_DIR, f"{model_u}_on_{data_u}.json")

        # Use cached JSON if it exists
        if os.path.exists(json_path):
            with open(json_path) as f:
                meta = json.load(f)
            matrix_data[model_u][data_u] = meta["accuracy"]
            print(f"  [{model_u} → {data_u}]  loaded from cache: {meta['accuracy']:.3f}")
            continue

        # Otherwise run inference
        data_path = ALL_USERS[data_u]
        if not os.path.exists(data_path):
            print(f"  ⚠ Data not found for {data_u}, skipping.")
            continue

        try:
            df  = load_data(data_path, EVAL_ROWS)
            df  = run_inference(df, tokenizer, model)
            acc = round(df["correct"].mean(), 4)
            matrix_data[model_u][data_u] = acc
            print(f"  [{model_u} → {data_u}]  accuracy: {acc:.3f}")

            # Save JSON cache
            n_correct = df["correct"].sum()
            n_total   = len(df)
            lo, hi    = proportion_confint(n_correct, n_total, method="wilson")
            present   = [l for l in LABEL_ORDER if l in df["true_grasp"].values]
            meta = {
                "model_user": model_u, "data_user": data_u,
                "accuracy":   acc,
                "wilson_ci_95":     [round(lo,4), round(hi,4)],
                "mean_confidence":  round(df["confidence"].mean(), 4),
                "correct": int(n_correct), "total": int(n_total),
                "classification_report": classification_report(
                    df["true_grasp"], df["predicted_grasp"],
                    labels=present, zero_division=0, output_dict=True),
            }
            with open(json_path, "w") as f:
                json.dump(meta, f, indent=4)
        except Exception as e:
            print(f"  ⚠ Error [{model_u} → {data_u}]: {e}")

# ── Build and display matrix ──────────────────────────────────────────────────
# ── Name anonymisation map ────────────────────────────────────────────────────
USER_LABELS = {
    "P2": "Participant_1",
    "P3":  "Participant_2",
    "P4": "Participant_3",
    "P1": "Participant_4",
    # "PilotA":    "Participant_5",
}
# acc_matrix = pd.DataFrame(matrix_data, index=user_list, columns=user_list).T
acc_matrix = pd.DataFrame(matrix_data, index=user_list, columns=user_list).T
acc_matrix = acc_matrix.rename(index=USER_LABELS, columns=USER_LABELS)
user_list_labeled = [USER_LABELS[u] for u in user_list]

print("\nCross-User Accuracy Matrix (rows=model trained on, cols=data evaluated on):")
display(acc_matrix.style
    .background_gradient(cmap="RdYlGn", vmin=0, vmax=1, axis=None)
    .format("{:.3f}", na_rep="N/A")
    .set_caption("Diagonal = same-user | Off-diagonal = cross-user")
)

fig, ax = plt.subplots(figsize=(max(6, len(user_list)*1.5), max(5, len(user_list)*1.2)))
sns.heatmap(
    acc_matrix.astype(float),
    annot=True, fmt=".3f", cmap="RdYlGn",
    vmin=0, vmax=1, linewidths=1.5, linecolor="white",
    ax=ax, cbar_kws={"label": "Accuracy"}, annot_kws={"size": 11}
)
# for i in range(len(user_list)):
for i in range(len(user_list_labeled)):
    ax.add_patch(plt.Rectangle((i, i), 1, 1, fill=False, edgecolor="black", lw=3))

ax.set_title("Cross-User Accuracy Matrix\n(rows = model trained on, columns = evaluated on)",
             fontsize=13, fontweight="bold", pad=14)
ax.set_xlabel("Data User (evaluated on)", fontsize=11)
ax.set_ylabel("Model User (trained on)", fontsize=11)
plt.xticks(rotation=30, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

_save(fig, "cross_user.svg")
# ── Stats ─────────────────────────────────────────────────────────────────────
# diagonal_vals  = [acc_matrix.loc[u, u] for u in user_list if not np.isnan(acc_matrix.loc[u, u])]
diagonal_vals = [acc_matrix.loc[u, u] for u in user_list_labeled if not np.isnan(acc_matrix.loc[u, u])]
off_diag_mask  = ~np.eye(len(user_list), dtype=bool)
off_diag_vals  = acc_matrix.values[off_diag_mask].astype(float)
off_diag_vals  = off_diag_vals[~np.isnan(off_diag_vals)]

if diagonal_vals:
    print(f"Same-user  (diagonal):     mean={np.mean(diagonal_vals):.3f}  min={np.min(diagonal_vals):.3f}  max={np.max(diagonal_vals):.3f}")
if len(off_diag_vals):
    print(f"Cross-user (off-diagonal): mean={np.mean(off_diag_vals):.3f}  min={np.min(off_diag_vals):.3f}  max={np.max(off_diag_vals):.3f}")

# Statistical Analysis Cells — cross_user_predictions.ipynb
Add these cells **after Cell 4 (Summary Table)** in your existing notebook.
They use `all_results` and `summary_df` already computed above.

---
## STAT CELL A · McNemar's Test — Diagonal vs Off-Diagonal Performance
Tests whether the personalised model performs significantly better on its **own user's data**
compared to other users' data (paired binary outcome per prompt).

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar
import itertools

print("=" * 70)
print("McNemar's Test: Model User vs Each Data User")
print("Tests whether classification accuracy differs significantly")
print("between the model's own-user data and other users' data.")
print("=" * 70)

user_ids = list(all_results.keys())

# We need a 'own user' result to compare against each cross-user result.
# MODEL_USER data must be in all_results for this test.
if MODEL_USER not in all_results:
    print(f"NOTE: MODEL_USER '{MODEL_USER}' not in DATA_USERS — McNemar test skipped.")
    print("To enable: add MODEL_USER to DATA_USERS dict in Cell 0 and re-run.")
else:
    own_correct = all_results[MODEL_USER]["correct"].values.astype(int)

    for uid in user_ids:
        if uid == MODEL_USER:
            continue
        other_correct = all_results[uid]["correct"].values.astype(int)
        n = min(len(own_correct), len(other_correct))
        a = own_correct[:n]; b = other_correct[:n]
        # Contingency table: [both correct, own only, other only, neither]
        c00 = int(np.sum((a == 1) & (b == 1)))
        c01 = int(np.sum((a == 1) & (b == 0)))
        c10 = int(np.sum((a == 0) & (b == 1)))
        c11 = int(np.sum((a == 0) & (b == 0)))
        table = [[c00, c01], [c10, c11]]
        result = mcnemar(table, exact=True)
        print(f"\n  {MODEL_USER} (own) vs {uid} (cross):")
        print(f"    Contingency table: {table}")
        print(f"    McNemar statistic = {result.statistic:.4f}")
        print(f"    p-value           = {result.pvalue:.4f}")
        sig = "SIGNIFICANT" if result.pvalue < 0.05 else "not significant"
        print(f"    -> {sig} at alpha = 0.05")

---
## STAT CELL B · Wilcoxon Signed-Rank Test — Diagonal vs Off-Diagonal Accuracy
Formally tests whether the within-user (diagonal) accuracy values in the cross-user matrix
are significantly higher than the off-diagonal (cross-user) values.
This is the key statistical validation of your personalisation claim.

In [ ]:
from scipy.stats import wilcoxon, mannwhitneyu
import warnings

# For this cell you need the full 5x5 cross-user accuracy matrix.
# Replace the values below with those from your Figure 10 matrix.
# Rows = model trained on, Columns = evaluated on
# Format: cross_user_matrix[trained_on][evaluated_on] = accuracy

print("=" * 70)
print("Wilcoxon Signed-Rank Test: Diagonal vs Off-Diagonal Accuracy")
print("=" * 70)

# --- PASTE YOUR CROSS-USER MATRIX VALUES HERE ---
# From Figure 10 in the paper (rows = trained on, cols = evaluated on)
cross_matrix = np.array([
    [0.795, 0.481, 0.667, 0.762, 0.881],   # User_1 model
    [0.421, 0.776, 0.479, 0.667, 0.693],   # User_2 model
    [0.464, 0.464, 0.824, 0.686, 0.836],   # User_3 model
    [0.376, 0.448, 0.481, 0.852, 0.886],   # User_4 model
    [0.462, 0.486, 0.619, 0.776, 0.967],   # User_5 model
])

n = cross_matrix.shape[0]
diagonal     = [cross_matrix[i, i] for i in range(n)]
off_diagonal = [cross_matrix[i, j] for i in range(n) for j in range(n) if i != j]

print(f"\n  Diagonal (within-user) accuracies : {[round(d,3) for d in diagonal]}")
print(f"  Mean diagonal                     : {np.mean(diagonal):.4f}")
print(f"  Off-diagonal accuracies (n={len(off_diagonal)}): {[round(o,3) for o in off_diagonal]}")
print(f"  Mean off-diagonal                 : {np.mean(off_diagonal):.4f}")

# Mann-Whitney U (diagonal vs off-diagonal; unpaired)
stat_mw, p_mw = mannwhitneyu(diagonal, off_diagonal, alternative="greater")
print(f"\n  Mann-Whitney U test (diagonal > off-diagonal):")
print(f"    U = {stat_mw:.4f},  p = {p_mw:.4f}")
print(f"    -> {'SIGNIFICANT' if p_mw < 0.05 else 'not significant'} at alpha = 0.05")

# Effect size: rank-biserial correlation
n1, n2 = len(diagonal), len(off_diagonal)
r_rb = 1 - (2 * stat_mw) / (n1 * n2)
print(f"    Rank-biserial correlation r = {r_rb:.4f} (effect size)")

---
## STAT CELL C · 95% Wilson Confidence Intervals — All Users (Enhanced Summary)
Adds confidence intervals to your existing summary table and prints
a clean, paper-ready results table.

In [ ]:
from statsmodels.stats.proportion import proportion_confint

print("=" * 70)
print("Cross-User Accuracy with 95% Wilson Confidence Intervals")
print("=" * 70)

fmt = "{:<12} {:>10} {:>14} {:>8} {:>8}"
print(fmt.format("Data User", "Accuracy", "95% CI", "Correct", "Total"))
print("-" * 70)

accs = []
for user_id, df in all_results.items():
    n_correct = int(df["correct"].sum())
    n_total   = len(df)
    acc       = n_correct / n_total
    lo, hi    = proportion_confint(n_correct, n_total, method="wilson")
    accs.append(acc)
    print(fmt.format(
        user_id,
        f"{acc:.4f}",
        f"[{lo:.4f}, {hi:.4f}]",
        str(n_correct),
        str(n_total)
    ))

print("-" * 70)
print(fmt.format("Mean", f"{np.mean(accs):.4f}", "", "", ""))
print(fmt.format("SD",   f"{np.std(accs):.4f}",  "", "", ""))
print("=" * 70)

---
## STAT CELL D · Cohen's Kappa — Per-User Classification Agreement
Measures how much better the model performs than chance for each user,
accounting for class imbalance. Values > 0.6 indicate substantial agreement.

In [ ]:
from sklearn.metrics import cohen_kappa_score

print("=" * 70)
print("Cohen's Kappa — Model Agreement vs Ground Truth (Per User)")
print("Interpretation: <0.2 slight | 0.2-0.4 fair | 0.4-0.6 moderate")
print("                0.6-0.8 substantial | >0.8 almost perfect")
print("=" * 70)

fmt = "{:<12} {:>10} {:>12}"
print(fmt.format("Data User", "Kappa", "Interpretation"))
print("-" * 40)

kappas = []
for user_id, df in all_results.items():
    kappa = cohen_kappa_score(df["true_grasp"], df["predicted_grasp"])
    kappas.append(kappa)
    if kappa > 0.8:   interp = "Almost perfect"
    elif kappa > 0.6: interp = "Substantial"
    elif kappa > 0.4: interp = "Moderate"
    elif kappa > 0.2: interp = "Fair"
    else:             interp = "Slight"
    print(fmt.format(user_id, f"{kappa:.4f}", interp))

print("-" * 40)
print(fmt.format("Mean", f"{np.mean(kappas):.4f}", ""))
print(fmt.format("SD",   f"{np.std(kappas):.4f}",  ""))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  STANDALONE: Base Model vs Adapted Model — Per-User Accuracy Comparison
#  Evaluated on the LAST 70 rows only (held-out test set)
#  No dependency on any other cell
# ══════════════════════════════════════════════════════════════════════════════
import os, torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ── CONFIGURE THESE ───────────────────────────────────────────────────────────
EXPERIMENTS_ROOT = r"D:\PythonEnvs\nlpglove\Continual_Learning_Experiments"
BASE_MODEL_PATH  = r"D:\PythonEnvs\nlpglove\bert_finetuned_updated"
EVAL_ROWS        = 70   # last N rows = held-out test set

ID2LABEL = {
    0: "grasp",       1: "open_hand",    2: "gentle_grasp",
    3: "tight_grasp", 4: "pinch_grasp",  5: "tighten",      6: "loosen"
}

ALL_USERS = {
    "P2": r"D:\PythonEnvs\nlpglove\Continual_Learning_Experiments/UserStudy_code_analysis/VoiceToGrasp_StudySheet_P2.xlsx",
    "P3":  r"D:\PythonEnvs\nlpglove\Continual_Learning_Experiments/UserStudy_code_analysis/VoiceToGrasp_StudySheet_P3.xlsx",
    "P4": r"D:\PythonEnvs\nlpglove\Continual_Learning_Experiments/UserStudy_code_analysis/VoiceToGrasp_StudySheet_P4.xlsx",
    "P1": r"D:\PythonEnvs\nlpglove\Continual_Learning_Experiments/UserStudy_code_analysis/VoiceToGrasp_StudySheet_P1.xlsx",
    "PilotA":    r"D:\PythonEnvs\nlpglove\Continual_Learning_Experiments/UserStudy_code_analysis/VoiceToGrasp_StudySheet_PilotA.xlsx",
}
# ─────────────────────────────────────────────────────────────────────────────

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}\n")

def load_last_n_rows(path, n):
    df = pd.read_excel(path) if path.endswith(".xlsx") else pd.read_csv(path)
    col_map = {}
    for col in df.columns:
        lc = col.strip().lower()
        if lc == "prompt":
            col_map[col] = "prompt"
        elif lc in ("grasp_type", "true_grasp"):
            col_map[col] = "true_grasp"
    df = df.rename(columns=col_map)
    df["prompt"]     = df["prompt"].astype(str).str.strip()
    df["true_grasp"] = df["true_grasp"].astype(str).str.strip()
    return df.iloc[-n:].reset_index(drop=True)   # ← last N rows

def run_inference(df, tokenizer, model):
    preds = []
    for text in df["prompt"]:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
        with torch.no_grad():
            logits = model(**inputs).logits
        pred_id = torch.argmax(logits, dim=1).item()
        preds.append(ID2LABEL[pred_id])
    df = df.copy()
    df["predicted"] = preds
    df["correct"]   = df["predicted"] == df["true_grasp"]
    return df

# ── Load base model once ──────────────────────────────────────────────────────
print("Loading base model...")
base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH)
base_model     = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL_PATH).to(device)
base_model.eval()
for p in base_model.parameters():
    p.requires_grad = False
print(f"  Base model loaded: {BASE_MODEL_PATH}\n")

# ── Evaluate each user ────────────────────────────────────────────────────────
results = []

for user, data_path in ALL_USERS.items():
    adapted_model_path = os.path.join(EXPERIMENTS_ROOT, user, "adaptive_model")

    if not os.path.exists(data_path):
        print(f"  ⚠ Data not found for {user}, skipping.")
        continue

    print(f"Evaluating: {user}")

    # Load held-out data
    df = load_last_n_rows(data_path, EVAL_ROWS)

    # Base model accuracy
    df_base = run_inference(df, base_tokenizer, base_model)
    acc_base = df_base["correct"].mean()
    print(f"  Base model accuracy:    {acc_base:.3f}")

    # Adapted model accuracy
    if os.path.exists(adapted_model_path):
        adapted_tokenizer = AutoTokenizer.from_pretrained(adapted_model_path)
        adapted_model     = AutoModelForSequenceClassification.from_pretrained(adapted_model_path).to(device)
        adapted_model.eval()
        for p in adapted_model.parameters():
            p.requires_grad = False

        df_adapted   = run_inference(df, adapted_tokenizer, adapted_model)
        acc_adapted  = df_adapted["correct"].mean()
        delta        = acc_adapted - acc_base
        print(f"  Adapted model accuracy: {acc_adapted:.3f}  (Δ {delta:+.3f})")

        # Free GPU memory
        del adapted_model, adapted_tokenizer
        torch.cuda.empty_cache()
    else:
        acc_adapted = np.nan
        delta       = np.nan
        print(f"  ⚠ Adapted model not found at {adapted_model_path}")

    results.append({
        "User":          user,
        "Base Acc":      round(acc_base, 4),
        "Adapted Acc":   round(acc_adapted, 4) if not np.isnan(acc_adapted) else np.nan,
        "Δ Accuracy":    round(delta, 4)        if not np.isnan(delta)       else np.nan,
    })

# ── Summary table ─────────────────────────────────────────────────────────────
# summary = pd.DataFrame(results).set_index("User")
summary = pd.DataFrame(results).set_index("User")
summary.index = summary.index.map(USER_LABELS)

def delta_color(val):
    if pd.isna(val): return ""
    color = "green" if val > 0 else ("red" if val < 0 else "gray")
    return f"color: {color}; font-weight: bold"

print("\n")
display(summary.style
    .background_gradient(subset=["Base Acc", "Adapted Acc"], cmap="RdYlGn", vmin=0, vmax=1, axis=None)
    .applymap(delta_color, subset=["Δ Accuracy"])
    .format("{:.3f}", na_rep="N/A")
    .set_caption(f"Base vs Adapted Model — Last {EVAL_ROWS} rows (held-out test set)")
)

# ── Bar chart ─────────────────────────────────────────────────────────────────
valid = summary.dropna(subset=["Base Acc", "Adapted Acc"])
users = valid.index.tolist()
x     = np.arange(len(users))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: grouped bar — base vs adapted
ax = axes[0]
bars_base    = ax.bar(x - width/2, valid["Base Acc"],    width, label="Base Model",    color="#5B8DB8", edgecolor="white")
bars_adapted = ax.bar(x + width/2, valid["Adapted Acc"], width, label="Adapted Model", color="#F4A340", edgecolor="white")

for bar in bars_base + bars_adapted:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(users, rotation=20, ha="right")
ax.set_ylim(0, 1.15)
ax.set_ylabel("Accuracy")
ax.set_title(f"Base vs Adapted Model\n(last {EVAL_ROWS} rows — held-out test set)")
ax.axhline(valid["Base Acc"].mean(),    color="#5B8DB8", linestyle="--", linewidth=1, alpha=0.7, label=f"Base mean={valid['Base Acc'].mean():.3f}")
ax.axhline(valid["Adapted Acc"].mean(), color="#F4A340", linestyle="--", linewidth=1, alpha=0.7, label=f"Adapted mean={valid['Adapted Acc'].mean():.3f}")
ax.legend(fontsize=9)

# Right: delta bar (improvement)
ax2 = axes[1]
colors = ["#2ecc71" if d >= 0 else "#e74c3c" for d in valid["Δ Accuracy"]]
bars_delta = ax2.bar(users, valid["Δ Accuracy"], color=colors, edgecolor="white", width=0.5)
for bar, val in zip(bars_delta, valid["Δ Accuracy"]):
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + (0.005 if val >= 0 else -0.018),
             f"{val:+.3f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

ax2.axhline(0, color="black", linewidth=0.8)
ax2.set_ylabel("Δ Accuracy (Adapted − Base)")
ax2.set_title("Accuracy Gain from Adaptation\n(green = improvement, red = regression)")
ax2.set_xticklabels(users, rotation=20, ha="right")

plt.tight_layout()
plt.show()